<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Workflow_Patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workflow Patterns: Chaining, Parallelization, Routing, and Orchestrator-Workers

Before you reach for an agent, reach for a **workflow**. A workflow is what you get when you stop asking one prompt to do five things and start writing the five steps yourself: LLM calls wired together with ordinary Python — sequential where order matters, concurrent where it doesn't, branching where the *kind* of request decides, and decomposed at runtime when the shape of the work is only knowable then.

Four patterns cover most of the LLM systems people actually ship, and this notebook builds them in the order they usually show up:

1. **Chaining** — each step's output is the next step's input.
2. **Parallelization** — independent steps run at the same time.
3. **Routing** — classify first, then dispatch to a specialist.
4. **Orchestrator-workers** — a planner LLM decides *what* the subtasks are, workers do them, a synthesizer merges the results.

And then the line that matters more than any of the four: **a workflow is a code path you fix in advance; an agent decides its own path at runtime.** Build the workflow first.

📎 *Section 11 is package territory. The LLM primitives — `generate()` and `extract()` — are imported from `tai-aitutor`, so the only thing we write by hand is the orchestration, which is the whole lesson.*

## 🧭 What You'll Learn

- Why one big prompt that does five things degrades — and why **decomposition** is the fix, not a better prompt
- **Chaining**: one job per call, with every intermediate artifact visible, cacheable, and testable
- **Parallelization** with `asyncio.gather` — why it buys **latency, not cost**, and why the concurrency cap is a feature
- **Routing**: one typed classification call, then a plain `if`/`else` into specialist prompts
- **Orchestrator-workers**: a Pydantic-typed plan written at runtime, workers run in parallel, a synthesizer merges
- A decision guide for picking a pattern — and the honest boundary between a workflow and an agent

## 1. Setup: Environment, Keys, and the Course Toolkit

The standard Section 9+ setup cell: the pinned course toolkit plus the pinned SDK profile, keys from Colab Secrets (🔑 icon) or a local `.env`, and the provider picker. Everything in this lesson runs on whichever `PROVIDER` you choose — the patterns are provider-neutral, which is exactly the point of building them out of one text primitive and one typed primitive.

The **model field is an editable dropdown** (`{allow-input: true}`): pick a listed course default or type any newer model ID straight into the box. (Locally, edit the string.)

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model provider (dropdown in Colab; edit the values locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]
CHAT_MODEL = "gemini-3.7-flash"  # @param ["gemini-3.7-flash", "gpt-5.6-luna", "claude-sonnet-5"] {allow-input: true}

_KEY_FOR = {"gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
REQUIRED_KEYS = [_KEY_FOR[PROVIDER]]

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # The course toolkit (exact-pinned) + the shared SDK profile (July 2026).
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "tai-aitutor[gemini,openai,anthropic]==0.0.3",
            "google-genai==2.3.0",
            "openai==2.46.0",
            "anthropic==0.117.0",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    # Locally: pip install "tai-aitutor[gemini,openai,anthropic]==0.0.3" once;
    # keys live in a .env file at the repo root.

from tai_aitutor import configure, setup_notebook

setup_notebook(required_keys=REQUIRED_KEYS)

cfg = configure(provider=PROVIDER, chat_model=CHAT_MODEL)

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | {cfg}")

✅ Setup complete — local | Config(provider='gemini', chat_model='gemini-3.7-flash', embed_provider='gemini', embed_model='gemini-embedding-001')


### The two primitives every pattern is built from

Two imports carry this entire notebook:

- `generate(prompt, system=...)` — 📎 *"How To Use LLMs via API"*: one prompt in, one string out.
- `extract(prompt, Schema)` — 📎 *the structured-outputs lesson*: one prompt in, a **validated Pydantic object** out, using each provider's native structured-output mechanism.

Every workflow pattern below is those two calls plus Python. That is not a simplification for teaching — it is what these systems are.

In [2]:
import time

from pydantic import BaseModel, Field

from tai_aitutor import extract, generate  # 📎 the text primitive and the typed primitive


def show(title, body):
    """Tiny console formatter, so intermediate artifacts are actually readable."""
    print(f"\n{'═' * 74}\n{title}\n{'─' * 74}")
    print(body if isinstance(body, str) else "\n".join(f"- {item}" for item in body))


print(generate("Reply with exactly: ready"))

ready


## 2. Why One Big Prompt Fails

Here is a real course chore: we have three lesson pages, and we want an **FAQ** for the course site — a set of questions a student would actually ask, each with an answer grounded *only* in those pages, each labelled with the pages it came from.

The obvious first attempt is one prompt that says all of that. Read it as a job list, though, and it is five jobs stacked in a single call: read three documents, invent N distinct questions, answer each one, keep every answer inside the sources, and attribute each answer correctly. The model has one forward pass to do all five, and quality on each one is negotiable against the others.

In [3]:
# Three (abbreviated) course pages — the raw material for the FAQ.
PAGES = [
    {
        "title": "What Retrieval-Augmented Generation Actually Does",
        "content": """
        Retrieval-Augmented Generation gives a language model access to documents it was never
        trained on. At query time the system embeds the question, searches a vector store for the
        most similar chunks, and pastes those chunks into the prompt alongside the question. The
        model then answers from text it can see rather than from memory, which is why RAG reduces
        hallucination on private or recent material and why a wrong answer is usually a retrieval
        failure rather than a generation failure. RAG also makes updates cheap: adding knowledge
        means adding documents to the store, not retraining a model.
        """,
    },
    {
        "title": "Chunking: Size, Overlap, and Why They Matter",
        "content": """
        Documents are split into chunks before they are embedded, because an embedding averages the
        meaning of everything inside it. Chunks that are too large dilute that meaning and drag
        irrelevant text into the prompt; chunks that are too small lose the context that made the
        passage answerable. Overlap repeats a little text between neighbouring chunks so that a
        sentence spanning a boundary still appears whole somewhere. Splitting on document structure
        — headings, sections, paragraphs — usually beats splitting on a fixed character count,
        because it keeps ideas that belong together in the same chunk.
        """,
    },
    {
        "title": "Evaluating a RAG Pipeline",
        "content": """
        A RAG pipeline is evaluated in two halves, because it can fail in two places. Retrieval is
        measured with ranking metrics over a set of questions whose correct chunk is known: hit rate
        asks whether the right chunk appeared at all, and mean reciprocal rank rewards putting it
        near the top. Generation is judged on the answer itself — faithfulness asks whether the
        answer is supported by the retrieved context, and relevancy asks whether it addresses the
        question. Evaluating the halves separately is what tells you which one to fix.
        """,
    },
]

COMBINED = "\n\n".join(f"Source Title: {p['title']}\nContent: {p['content'].strip()}" for p in PAGES)
print(COMBINED[:400], "...")

Source Title: What Retrieval-Augmented Generation Actually Does
Content: Retrieval-Augmented Generation gives a language model access to documents it was never
        trained on. At query time the system embeds the question, searches a vector store for the
        most similar chunks, and pastes those chunks into the prompt alongside the question. The
        model then answers from text it can s ...


Now the single call. A typed schema is doing real work here — the reply is guaranteed to *parse* — but a schema constrains the **shape** of the output, never its **content**. Nothing in `FAQList` can make the questions distinct, the answers grounded, or the attributions true.

In [4]:
class FAQ(BaseModel):
    """One question, its answer, and the sources the answer came from."""

    question: str = Field(description="The question a reader would ask.")
    answer: str = Field(description="A concise answer, taken only from the provided content.")
    sources: list[str] = Field(description="The exact Source Titles used to write the answer.")


class FAQList(BaseModel):
    faqs: list[FAQ] = Field(description="The generated FAQ entries.")


N_QUESTIONS = 4

ONE_BIG_PROMPT = f"""
From the course content below, produce exactly {N_QUESTIONS} frequently asked questions.
For each question write a concise answer derived ONLY from the text, and list the exact
'Source Title' values you used for that specific answer. The questions must be distinct
and must not overlap.

<provided_content>
{{content}}
</provided_content>
""".strip()

one_shot = extract(ONE_BIG_PROMPT.format(content=COMBINED), FAQList)

for faq in one_shot.faqs:
    show(faq.question, f"{faq.answer}\n\nsources: {faq.sources}")


══════════════════════════════════════════════════════════════════════════
How does Retrieval-Augmented Generation (RAG) work at query time?
──────────────────────────────────────────────────────────────────────────
At query time, the system embeds the question, searches a vector store for the most similar chunks, and pastes those chunks into the prompt alongside the question so the model answers from visible text rather than memory.

sources: ['What Retrieval-Augmented Generation Actually Does']

══════════════════════════════════════════════════════════════════════════
Why does RAG make updating knowledge cost-effective?
──────────────────────────────────────────────────────────────────────────
RAG makes updates cheap because adding new knowledge only requires adding documents to the store rather than retraining the language model.

sources: ['What Retrieval-Augmented Generation Actually Does']

══════════════════════════════════════════════════════════════════════════
Why is overla

**What just happened?** Something plausible, probably — that is what makes this failure mode expensive. Read the output against the five jobs and check each one separately: are there exactly `N_QUESTIONS`? Are any two of them the same question in different words? Does every answer stay inside the three pages? And the one worth staring at: are the `sources` the pages the answer actually used, or a reasonable-looking list produced *after* the answer was written, which the model has no mechanism to verify?

The deeper problem is that you cannot tell. There is one artifact — the final list — and no intermediate step to inspect, no place to insert a check, nothing to cache, and no way to improve the attribution prompt without editing the same paragraph that controls question generation. One temperature roll decides all five jobs at once.

The fix is not a better prompt. It is fewer jobs per prompt.

## 3. Chaining: One Job Per Call

Chaining is the simplest pattern and the one you will use most: split the task into steps, give each step its own call with its own prompt, and let each step's output feed the next.

Our FAQ chore becomes three steps — **generate questions → answer each question → attribute the answer**. The order is not arbitrary: you cannot answer a question you have not asked, and you cannot say which sources an answer used before the answer exists. That dependency is what makes this a chain rather than three independent calls.

### Step 1 — ask the questions

One job: read the material and produce good, distinct questions. No answering, no attribution. The typed return means the next step gets a `list[str]` instead of a numbered list it has to parse.

In [5]:
class QuestionList(BaseModel):
    """The questions a reader would ask about this material."""

    questions: list[str] = Field(description="Distinct, self-contained questions.")


QUESTIONS_PROMPT = """
Read the course content below and write exactly {n} distinct questions a student would
realistically ask about it. Each question must stand on its own and must be answerable
from the content. Do not answer them.

<provided_content>
{content}
</provided_content>
""".strip()


def generate_questions(content: str, n: int = N_QUESTIONS) -> list[str]:
    """Step 1 of the chain: material in, questions out."""
    return extract(QUESTIONS_PROMPT.format(n=n, content=content), QuestionList).questions


questions = generate_questions(COMBINED)
show("Step 1 — questions", questions)


══════════════════════════════════════════════════════════════════════════
Step 1 — questions
──────────────────────────────────────────────────────────────────────────
- Why does Retrieval-Augmented Generation make updating a model's knowledge cheaper than retraining?
- What are the potential drawbacks of creating text chunks that are either too large or too small during embedding?
- What is the purpose of adding overlap between neighboring chunks when preparing documents?
- Which distinct metrics are used to separately evaluate the retrieval and generation stages of a RAG pipeline?


### Step 2 — answer one question

One job again, and a narrower one: this call sees a single question and the source material, and it is told in as many words that anything not in the material does not exist. A prompt this small is easy to tune, and tuning it cannot damage step 1.

In [6]:
ANSWER_PROMPT = """
Using ONLY the provided content, answer the question below in two or three sentences.
If the content does not contain the answer, say exactly: "Not covered in the provided material."

<question>
{question}
</question>

<provided_content>
{content}
</provided_content>
""".strip()


def answer_question(question: str, content: str) -> str:
    """Step 2 of the chain: one question in, one grounded answer out."""
    return generate(ANSWER_PROMPT.format(question=question, content=content))


first_answer = answer_question(questions[0], COMBINED)
show(f"Step 2 — answer to: {questions[0]}", first_answer)


══════════════════════════════════════════════════════════════════════════
Step 2 — answer to: Why does Retrieval-Augmented Generation make updating a model's knowledge cheaper than retraining?
──────────────────────────────────────────────────────────────────────────
Retrieval-Augmented Generation makes updates cheaper because adding knowledge only requires adding documents to the vector store rather than retraining a model. At query time, the system searches the store for relevant chunks and pastes them directly into the prompt for the model to reference.


### Step 3 — attribute the answer

The step the single big prompt could never do honestly. Attribution now happens *after* the answer exists, as its own call that reads the finished answer and the sources and decides which ones support it. Same model, same material — but it is judging a text it can see instead of predicting its own citation while still writing.

In [7]:
class SourceList(BaseModel):
    """The source titles an answer actually relied on."""

    sources: list[str] = Field(description="Exact 'Source Title' values supporting the answer.")


SOURCES_PROMPT = """
Below are a question, an answer written from a set of documents, and those documents.
Decide which documents actually support the answer. Return their exact 'Source Title' values,
and nothing that merely looks related.

<question>
{question}
</question>

<answer>
{answer}
</answer>

<provided_content>
{content}
</provided_content>
""".strip()


def find_sources(question: str, answer: str, content: str) -> list[str]:
    """Step 3 of the chain: a finished answer in, its real sources out."""
    return extract(
        SOURCES_PROMPT.format(question=question, answer=answer, content=content), SourceList
    ).sources


show("Step 3 — sources", find_sources(questions[0], first_answer, COMBINED))


══════════════════════════════════════════════════════════════════════════
Step 3 — sources
──────────────────────────────────────────────────────────────────────────
- What Retrieval-Augmented Generation Actually Does


### The chain

Three functions, one loop, and the intermediate artifacts printed as they appear — because **visibility is the point of this pattern**. Every arrow between two steps is a Python variable you can print, log, cache, assert on, or replace with a different model.

In [8]:
def faq_chain(content: str, n: int = N_QUESTIONS, verbose: bool = True) -> list[FAQ]:
    """generate_questions → answer_question → find_sources, one question at a time."""
    faqs = []
    for question in generate_questions(content, n):
        answer = answer_question(question, content)  # step 2 needs step 1
        sources = find_sources(question, answer, content)  # step 3 needs step 2
        if verbose:
            show(f"Q: {question}", f"{answer}\n\nsources: {sources}")
        faqs.append(FAQ(question=question, answer=answer, sources=sources))
    return faqs


start = time.perf_counter()
chained_faqs = faq_chain(COMBINED)
chain_seconds = time.perf_counter() - start

print(f"\nChain: {len(chained_faqs)} FAQs in {chain_seconds:.1f}s, {1 + 2 * len(chained_faqs)} model calls")


══════════════════════════════════════════════════════════════════════════
Q: Why does Retrieval-Augmented Generation (RAG) reduce hallucinations when dealing with private or recent material?
──────────────────────────────────────────────────────────────────────────
Retrieval-Augmented Generation reduces hallucinations because it searches for the most similar document chunks and pastes them directly into the prompt alongside the question. This allows the language model to answer from text it can actively see rather than relying on memory.

sources: ['What Retrieval-Augmented Generation Actually Does']



══════════════════════════════════════════════════════════════════════════
Q: What problems occur if document chunks are made either too large or too small before embedding?
──────────────────────────────────────────────────────────────────────────
If document chunks are too large, they dilute the meaning of the embedding and drag irrelevant text into the prompt. Conversely, if chunks are too small, they lose the context that makes the passage answerable.

sources: ['Chunking: Size, Overlap, and Why They Matter']



══════════════════════════════════════════════════════════════════════════
Q: What is the purpose of using overlap when splitting documents into chunks?
──────────────────────────────────────────────────────────────────────────
The purpose of using overlap is to repeat a little text between neighbouring chunks. This ensures that a sentence spanning a boundary still appears whole somewhere.

sources: ['Chunking: Size, Overlap, and Why They Matter']



══════════════════════════════════════════════════════════════════════════
Q: Which specific metrics are used to evaluate the retrieval and generation stages of a RAG pipeline?
──────────────────────────────────────────────────────────────────────────
The retrieval stage is evaluated using ranking metrics, specifically hit rate to determine if the correct chunk appeared and mean reciprocal rank to reward placing it near the top. The generation stage is evaluated using faithfulness, which assesses whether the answer is supported by the retrieved context, and relevancy, which checks whether the answer addresses the question.

sources: ['Evaluating a RAG Pipeline']

Chain: 4 FAQs in 26.1s, 9 model calls


**What just happened?** The same five jobs got done, but each call did one of them, and you watched the work accumulate. That changes what you can do about it:

- **Debug**: a weak FAQ has a first bad step, and you can see which one it was.
- **Tune independently**: rewriting the attribution prompt cannot make the questions worse.
- **Insert checks between steps**: drop near-duplicate questions before you pay to answer them; refuse to publish an FAQ whose `sources` came back empty.
- **Mix models**: a small cheap model can generate questions while a stronger one writes answers — the chain does not care, because the interface between steps is a Python value.

The bill for all that is real: more calls, more tokens (the content is re-sent on every step), and more wall-clock time. That last cost is the next section's problem.

## 4. Parallelization: Independent Work Should Not Queue

Look again at the loop. Steps 2 and 3 are genuinely dependent — attribution reads the answer. But question 2's answer does not depend on question 1's answer in any way. Those are independent units of work, and independent LLM calls have no business waiting in line.

The reason concurrency helps so much here is that an LLM call is almost entirely *waiting*: your process sends a request and then sits idle while the model generates. Overlap the waiting and the wall clock collapses toward the slowest single call.

Be precise about what improves: **latency, not cost.** The same calls are made, the same tokens are sent and generated, and the bill is identical — they simply overlap in time. (If you want the *price* to drop, that is a different lever: the provider Batch APIs trade latency for a discount, 📎 priced in the applied-structured-outputs lesson.)

And cap the concurrency. Providers enforce per-minute request limits, so an uncapped `gather` over two hundred questions is a 429 generator. `asyncio.Semaphore` makes the ceiling explicit and deliberate.

[NEEDS UPDATE — current requests-per-minute limit for the course-default model on the free tier | Google AI Studio rate-limit docs (ai.google.dev/gemini-api/docs/rate-limits) | this is the number that sets a sensible MAX_CONCURRENT, and it is where students hit their first 429]

In [9]:
import asyncio

MAX_CONCURRENT = 4  # the cap is the point: stay inside your rate limit ON PURPOSE

# generate() and extract() are ordinary blocking functions, so asyncio.to_thread hands
# each call to a worker thread and gives the event loop something to await.


async def faq_for(question: str, content: str, semaphore: asyncio.Semaphore) -> FAQ:
    """One question end to end. Steps 2 and 3 stay sequential — step 3 reads step 2."""
    async with semaphore:  # at most MAX_CONCURRENT questions in flight
        answer = await asyncio.to_thread(answer_question, question, content)
        sources = await asyncio.to_thread(find_sources, question, answer, content)
    return FAQ(question=question, answer=answer, sources=sources)


async def faq_parallel(content: str, n: int = N_QUESTIONS) -> list[FAQ]:
    """Step 1 first (everything depends on it), then all questions concurrently."""
    questions = await asyncio.to_thread(generate_questions, content, n)
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    return list(await asyncio.gather(*(faq_for(q, content, semaphore) for q in questions)))

`await` at the top level of a cell works in both Colab and Jupyter — the notebook kernel already runs an event loop. That is also why `asyncio.run()` raises there, and why no notebook in this course needs one of those event-loop patching cells you may have seen at the top of older material.

Run it and compare the two numbers **your** run prints. Both workflows made the same calls with the same prompts.

In [10]:
start = time.perf_counter()
parallel_faqs = await faq_parallel(COMBINED)  # top-level await: Colab and Jupyter both support it
parallel_seconds = time.perf_counter() - start

print(f"Chain (sequential): {chain_seconds:6.1f}s")
print(f"Parallel (cap={MAX_CONCURRENT}):  {parallel_seconds:6.1f}s")
print(f"Same {1 + 2 * len(parallel_faqs)} calls, same prompts, same tokens — only the waiting overlapped.")

show("First parallel FAQ", f"{parallel_faqs[0].question}\n\n{parallel_faqs[0].answer}")

Chain (sequential):   26.1s
Parallel (cap=4):    10.4s
Same 9 calls, same prompts, same tokens — only the waiting overlapped.

══════════════════════════════════════════════════════════════════════════
First parallel FAQ
──────────────────────────────────────────────────────────────────────────
Why is an incorrect answer in a Retrieval-Augmented Generation (RAG) system usually considered a retrieval failure rather than a generation failure?

In a RAG system, the language model answers directly from text it can see rather than from its memory, as retrieved chunks are pasted into the prompt alongside the question. Because the model relies on this provided context to produce its response, an incorrect answer is typically caused by a failure to retrieve the proper text rather than a generation error.


**What just happened?** Nothing about the work changed; only its scheduling did. Wall clock now scales roughly with `ceil(n / MAX_CONCURRENT)` batches of one call's latency instead of `n` of them — read your own two numbers rather than trusting any published ratio, because it depends on your model, your prompt lengths, and your network.

Three details worth keeping:

- **`gather` preserves order.** Results come back in the order you passed the coroutines in, not the order they finished, so `parallel_faqs[i]` still corresponds to `questions[i]`.
- **Dependencies must not be parallelized.** Steps 2 and 3 stay `await`-ed in sequence inside each task. Parallelism is legal exactly where there is no data flow between the units.
- **One failure sinks the whole `gather`** by default. `asyncio.gather(..., return_exceptions=True)` returns the exceptions as results instead, so nineteen good answers survive one bad one — worth doing the moment this runs unattended.

📎 *For plain single-prompt fan-outs, a `concurrent.futures.ThreadPoolExecutor` over `generate()` is the same idea with less machinery — the toolkit deliberately ships no batching wrapper. Reach for `asyncio` when the units are multi-step, like ours.*

## 5. Routing: Classify First, Then Dispatch

Chaining and parallelization assume every request needs the same treatment. Real inboxes do not work that way. A student asking why their Chroma query returns duplicates, a student asking what an embedding *is*, and a student asking when assignment 3 is due need three different prompts — and a single prompt tuned to serve all three serves none of them well.

Routing is the fix, and it is smaller than its reputation: **one typed classification call, then a plain `if`/`else` into a specialist prompt.** No router engine, no selector classes.

📎 *This is exactly the shape of the routing lesson in Section 9 — the same `route()` call, the same `if`/`else` — except there it chose between knowledge bases and models inside a RAG pipeline. Here it is the general pattern: routing is a workflow primitive, and RAG is one place you apply it.*

In [11]:
from tai_aitutor import route  # 📎 Section 9: routing as ONE typed classification call

STUDENT_ROUTES = {
    "debug": (
        "Something the student built is broken or behaving unexpectedly: errors, stack traces, "
        "wrong output, code that runs but returns nonsense."
    ),
    "concept": (
        "The student wants to understand an idea from the course — what something is, how it "
        "works, why one approach beats another."
    ),
    "logistics": (
        "Questions about the course itself: deadlines, where a notebook lives, how grading works, "
        "which lesson covers what."
    ),
}

MESSAGES = [
    "My Chroma query returns the same chunk five times no matter what I ask. What did I break?",
    "I still don't get why we overlap chunks. Doesn't that just duplicate text?",
    "Which lesson has the notebook for the evaluation section, and is it graded?",
]

for message in MESSAGES:
    decision = route(message, routes=STUDENT_ROUTES)
    print(f"[{decision.route}] {decision.reason}\n  ↳ {message}\n")

[debug] The student's Chroma query is behaving unexpectedly by returning identical duplicate chunks, indicating a bug in their implementation.
  ↳ My Chroma query returns the same chunk five times no matter what I ask. What did I break?



[concept] The student is asking for a conceptual explanation of why chunk overlapping is used rather than seeking debugging help or course logistics.
  ↳ I still don't get why we overlap chunks. Doesn't that just duplicate text?



[logistics] The question asks about the location of a specific notebook and whether an assignment is graded, which pertains to course organization and logistics.
  ↳ Which lesson has the notebook for the evaluation section, and is it graded?



The classification is one cheap call and its verdict is a validated string, so the dispatch is the least interesting code in this notebook — which is how you know the pattern is right. Each branch owns a system prompt written for exactly one kind of request.

In [12]:
DEBUG_SYSTEM = (
    "You are a debugging assistant for an applied LLM course. Ask for the smallest missing detail "
    "if you need it, name the most likely cause first, and give one concrete thing to try next."
)
CONCEPT_SYSTEM = (
    "You are a teacher on an applied LLM course. Explain the idea in plain language, use one "
    "concrete example, and end with the trade-off a practitioner actually faces."
)
LOGISTICS_SYSTEM = (
    "You are a course assistant. Answer administrative questions briefly. If you do not have the "
    "specific detail, say so plainly and point the student at the course page rather than guessing."
)


def handle(message: str) -> str:
    """Routing in full: one classification call, then plain-Python dispatch."""
    decision = route(message, routes=STUDENT_ROUTES)

    if decision.route == "debug":
        system = DEBUG_SYSTEM
    elif decision.route == "concept":
        system = CONCEPT_SYSTEM
    else:
        system = LOGISTICS_SYSTEM

    print(f"[route: {decision.route}] {decision.reason}")
    return generate(message, system=system)


show(MESSAGES[1], handle(MESSAGES[1]))

[route: concept] The student is asking for a conceptual explanation of why chunk overlap is used rather than seeking debugging help or course logistics.



══════════════════════════════════════════════════════════════════════════
I still don't get why we overlap chunks. Doesn't that just duplicate text?
──────────────────────────────────────────────────────────────────────────
Yes, you are 100% right: it *does* duplicate text. But that duplication is an intentional safety net.

When we split documents into chunks, we use arbitrary cutoffs (like every 500 words). The computer doesn't know where a complete thought begins or ends. If you cut text cleanly with zero overlap, you inevitably slice critical ideas in half, which breaks both the **search** (the embedding) and the **answer** (the LLM's comprehension).

### A Concrete Example

Imagine your source text contains this:

> *"Project Apollo was a massive success. However, because of severe budget cuts in phase two, the team was forced to lay off 40% of its staff."*

If you split right down the middle with **no overlap**:
* **Chunk 1:** *"Project Apollo was a massive success. However, be

### When the branch needs more than a name

`route()` returns a route and a reason, which is all a two-way switch needs. When the branch needs to *act* on more — an urgency level for the queue, a cleaned-up restatement for the specialist — classify with `extract()` and a `Literal` schema instead. That is all `route()` is underneath: one `extract()` call against a fixed schema.

In [13]:
from typing import Literal


class Triage(BaseModel):
    """A route plus everything the branch needs to act on it."""

    lane: Literal["debug", "concept", "logistics"] = Field(description="Which specialist handles this.")
    urgency: Literal["low", "normal", "blocking"] = Field(description="Is the student stuck right now?")
    restated: str = Field(description="The request restated as one clear sentence for the specialist.")


triage = extract(f"Triage this message from a student.\n\n{MESSAGES[0]}", Triage)
print(triage.model_dump_json(indent=2))

{
  "lane": "debug",
  "urgency": "blocking",
  "restated": "The student needs help debugging a Chroma retrieval query that returns the same chunk five times regardless of the query input."
}


**What just happened?** The decision was made **before** any expensive work started, by a call that reads one short message instead of a whole pipeline's context. That has three consequences worth naming:

- **It is cheap.** Classification is a tiny prompt and a tiny output. You pay it once per request to avoid running the wrong branch.
- **It is auditable.** `decision.route` and `decision.reason` are values you can log next to the request. Months later, "why did it answer like that?" is a database query, not an archaeology project.
- **It is fixable in prose.** When a message lands in the wrong lane, the bug is almost always the route *description*, not the model. The model chooses by reading those descriptions — write them like tool descriptions, because that is what they are.

The cost is that exactly one branch runs. A request that genuinely needs two specialists gets one — which is precisely where the next pattern starts.

## 6. Orchestrator-Workers: a Plan Written at Runtime

Chaining, parallelization and routing all fix the shape of the work in advance. You wrote the steps, you wrote the branches, and every request flows through one of the paths you already drew.

Some requests refuse to fit. A single student message can contain a bug report, a conceptual confusion, and a "what should I do next?" — three different jobs, and the *number and kind* of jobs changes with every message. That is what the orchestrator-workers pattern is for:

1. an **orchestrator** call reads the request and emits a typed list of tasks — the plan;
2. your code dispatches each task to a **worker** with a specialist prompt, in parallel where the tasks are independent;
3. a **synthesizer** call merges the worker outputs into one answer for the human.

The Pydantic types are load-bearing. The plan is validated *before* any worker runs, so a malformed plan fails at the boundary, cheaply, instead of halfway through the expensive part. And because `kind` is a `Literal`, a task naming a worker that does not exist cannot survive validation — there is no "unknown task type" branch to write.

In [14]:
class WorkerTask(BaseModel):
    """One unit of work the orchestrator wants done."""

    kind: Literal["explain_concept", "review_code", "plan_next_steps"] = Field(
        description="Which specialist worker should handle this task."
    )
    instruction: str = Field(description="A self-contained instruction for that worker.")
    context: str = Field(
        default="", description="Any part of the student's message the worker needs verbatim."
    )


class Plan(BaseModel):
    """The orchestrator's decomposition of one request."""

    tasks: list[WorkerTask] = Field(description="One task per distinct thing the student asked for.")


ORCHESTRATOR_SYSTEM = """
You plan work for a course support team. Break the student's message into the smallest set of
independent tasks that together answer it completely — one task per distinct ask, no duplicates,
no invented work. Available workers:

- explain_concept: explains an idea from the course.
- review_code: reads a code snippet and finds the problem.
- plan_next_steps: recommends what the student should do or study next.

Each task must be understandable on its own, without the original message.
""".strip()


def orchestrate(request: str) -> list[WorkerTask]:
    """The planner: one request in, a validated task list out."""
    return extract(f"Student message:\n\n{request}", Plan, system=ORCHESTRATOR_SYSTEM).tasks

### The workers

A worker is a specialist prompt and one call — nothing more. They are separate functions for the same reason the chain's steps were: each prompt has one job, so each can be tuned, tested, or pointed at a different model on its own.

In [15]:
WORKER_SYSTEMS = {
    "explain_concept": (
        "You explain one idea from an applied LLM course. Plain language, one concrete example, "
        "and the practical trade-off. Three short paragraphs at most."
    ),
    "review_code": (
        "You review a short Python snippet from an LLM/RAG pipeline. Name the most likely bug, "
        "explain why it produces the reported symptom, and show the corrected lines only."
    ),
    "plan_next_steps": (
        "You recommend what a student should do next. Two or three concrete steps, ordered, each "
        "with one sentence on why it comes at that point."
    ),
}


def run_worker(task: WorkerTask) -> str:
    """One worker = one specialist system prompt + one call."""
    # Literal-typed `kind` means this lookup cannot miss: an unknown worker
    # name would have failed Pydantic validation before we ever got here.
    prompt = f"{task.instruction}\n\n---\n{task.context}" if task.context else task.instruction
    return generate(prompt, system=WORKER_SYSTEMS[task.kind])

### Workers in parallel, then one synthesizer

The orchestrator was told to emit *independent* tasks, so the workers can run concurrently — the same `gather`-with-a-cap from Section 4, now over a task list whose length nobody hard-coded. The synthesizer then does the job the workers deliberately did not: turn three specialist outputs into one reply that reads as though a person wrote it.

In [16]:
SYNTHESIZER_SYSTEM = """
You write the final reply to a student, merging notes from several specialists into one coherent
answer. Address every part of their message in a sensible order, keep the technical content exactly
as the specialists wrote it, remove repetition, and sound like one helpful human rather than three
appended sections. No preamble about the process.
""".strip()


async def run_workers(tasks: list[WorkerTask]) -> list[str]:
    """Independent tasks, so: same capped gather as the parallelization section."""
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)

    async def one(task: WorkerTask) -> str:
        async with semaphore:
            return await asyncio.to_thread(run_worker, task)

    return list(await asyncio.gather(*(one(task) for task in tasks)))


def synthesize(request: str, tasks: list[WorkerTask], results: list[str]) -> str:
    """The merge step: specialist outputs in, one human-facing answer out."""
    notes = "\n\n".join(
        f"[{task.kind}] {task.instruction}\n{result}" for task, result in zip(tasks, results)
    )
    return generate(
        f"Student message:\n{request}\n\nSpecialist notes:\n{notes}", system=SYNTHESIZER_SYSTEM
    )


async def handle_request(request: str) -> str:
    """orchestrator → workers (parallel) → synthesizer, with every stage visible."""
    tasks = orchestrate(request)
    show("Plan", [f"{task.kind}: {task.instruction}" for task in tasks])

    results = await run_workers(tasks)
    for task, result in zip(tasks, results):
        show(f"Worker — {task.kind}", result)

    return synthesize(request, tasks, results)

One message with three asks buried in it. Watch the plan appear first — nothing in our code decided there would be three tasks, or which kinds they would be.

In [17]:
STUDENT_MESSAGE = """
Hi! Two things and a question. First, my retrieval got worse after I switched to bigger chunks
and I don't really understand why — I thought more context was strictly better?

Second, I think my ingest loop is wrong; every time I re-run it the collection doubles:

    for chunk in chunks:
        collection.add(
            ids=[str(uuid.uuid4())], documents=[chunk], metadatas=[{"doc": doc_title}]
        )

And finally, I've finished the retrieval lessons and I'm not sure whether to do evaluation next
or jump to the agent material. What would you suggest?
""".strip()

final_reply = await handle_request(STUDENT_MESSAGE)
show("Final synthesized reply", final_reply)


══════════════════════════════════════════════════════════════════════════
Plan
──────────────────────────────────────────────────────────────────────────
- explain_concept: Explain why larger chunk sizes can degrade vector retrieval performance, addressing the misconception that more context is strictly better.
- review_code: Identify why the collection doubles in size every time the ingestion loop runs and explain how to prevent this duplicate insertion behavior.
- plan_next_steps: Recommend whether the student should proceed to evaluation or move directly to the agent material after completing the retrieval lessons.



══════════════════════════════════════════════════════════════════════════
Worker — explain_concept
──────────────────────────────────────────────────────────────────────────
Larger chunks degrade retrieval because of **semantic dilution**. An embedding model compresses an entire chunk into a single, fixed-size vector. When you pack multiple ideas into one large chunk, its embedding becomes a blurry "average" of all those topics rather than a sharp match for any specific one. While more context helps the LLM *generate* a good answer, stuffing too much text into a single chunk creates noise that prevents the search engine from *finding* it.

For example, imagine a 2,000-word document covering your product's refund policy, API rate limits, and database configuration. If a user asks, *"How do I get my money back?"*, the embedding for that massive chunk is pulled in three different directions and scores a weak similarity match. In contrast, a 100-word chunk focused solely on the refund po


══════════════════════════════════════════════════════════════════════════
Final synthesized reply
──────────────────────────────────────────────────────────────────────────
### 1. Why Larger Chunks Degraded Retrieval

The reason your retrieval suffered comes down to **semantic dilution**. 

An embedding model compresses an entire chunk of text into a single, fixed-size vector. When you pack multiple ideas into one large chunk, its embedding becomes a blurry "average" of all those topics rather than a sharp match for any single concept. While having more context is great for helping an LLM *generate* a detailed answer, stuffing too much text into one chunk introduces noise that prevents the vector search engine from *finding* it in the first place.

For example, imagine a 2,000-word document covering a product's refund policy, API rate limits, and database configuration. If a user asks, *"How do I get my money back?"*, the embedding for that massive chunk is pulled in three different 

**What just happened?** Three layers did three different jobs, and only the middle one looked like the patterns you already knew.

The **orchestrator** decided the shape of the work. Nothing in our code said "there are three tasks here" — a different message would have produced one task, or five, of different kinds. That is the whole reason to accept this pattern's extra complexity: routing picks a path from a menu you wrote, while an orchestrator *writes the path*, within the vocabulary your `Literal` allows.

The **workers** ran concurrently, using the section-4 machinery unchanged, because the orchestrator was instructed to emit independent tasks. If your task types can depend on each other, that instruction is a lie and the `gather` is a bug — you would need to run them in dependency waves instead.

The **synthesizer** is not optional garnish. Three specialist outputs concatenated read like three specialist outputs concatenated; the merge call is what makes the reply feel like one answer, and it is also where you enforce tone, length, and format once for the whole system.

Two honest costs. First, arithmetic: one plan + N workers + one synthesis is N+2 calls and one long final prompt for a single reply. This pattern earns its keep when the subtasks genuinely differ; it is expensive theatre when they do not. Second, there is no feedback: the plan is written once, before any worker has produced anything, and nothing here re-plans when a worker comes back empty. Adding that loop is exactly what turns this workflow into an agent — which is the next thing to be precise about.

## 7. Choosing a Pattern — and the Line Where Agents Begin

Start with the simplest thing that fits the work:

| If the work… | Use | Shape |
|---|---|---|
| is one job | **a single call** | prompt → answer |
| is several jobs, each needing the previous one's output | **chaining** | a → b → c |
| is many independent units of the same job | **parallelization** | `gather(a, a, a)` with a cap |
| arrives in distinct kinds that deserve different handling | **routing** | classify → `if`/`else` |
| decomposes differently for every request | **orchestrator-workers** | plan → workers → synthesize |
| genuinely cannot be planned before it starts | **an agent** | model decides → tool runs → repeat |

Patterns compose, and in production they always do: our orchestrator-workers pipeline contains a parallelization; a routed branch is usually a chain; a chain step can be a routed sub-workflow. Nothing here is exclusive.

### The boundary: workflows vs agents

**A workflow is a code path you fix in advance. An agent decides its own path at runtime.**

Notice that the orchestrator does not blur this line, even though a model wrote the plan. The *control flow* stayed ours: our code called the orchestrator, our loop dispatched the tasks, our `Literal` bounded which workers could ever run, our function called the synthesizer. The model filled in a plan; it never decided what happens next. Every run takes the same three-stage path, and you can draw that path on a whiteboard before the request arrives.

An agent hands the control flow itself to the model. It looks at the result of the last tool call and chooses the next action — another tool, a different tool, or an answer — and keeps choosing until it decides it is finished. Nobody can draw that path in advance, because it does not exist until the run happens.

That is a real trade, not a hierarchy:

- **Workflows** give you predictable latency and cost, traces that read the same way every time, tests that mean something, and failures that localize to a step. The price is that they only handle work whose shape you anticipated.
- **Agents** handle work whose shape you could not anticipate. The price is unbounded loops, unbounded cost, and behaviour that varies run to run.

So: **build the workflow first.** Most tasks that look like they need an agent are a chain with a router in front of it. Reach for an agent when the path genuinely cannot be known ahead of time — and when you do, keep these patterns *inside* it, because an agent's individual steps are still chains, parallel fan-outs, and routed calls.

⏭️ *Next lesson — **LLM Agents**: the Thought → Action → Observation loop, native tool calling in all three provider APIs, and the two safety rails (never `eval()` model output; always bound the loop) that a fixed workflow gives you for free.*

⏭️ *Section 12 — **LangGraph**: these same patterns expressed as a graph, where chaining is an edge, routing is a conditional edge, parallelization is a fan-out/fan-in, and orchestrator-workers is a supervisor node — plus the persistence, streaming, and human-in-the-loop pauses that a hand-written pipeline would have to grow itself.*

## 🎯 Your Turn

**Easy — extend the router.** Add a fourth route, `resources` ("asks for a link, a paper, a dataset, or further reading"), with its own specialist prompt and its own `elif` branch. Send three new messages through `handle()`, including one deliberately ambiguous ("where can I read more about why my chunks are wrong?"). When it lands in the wrong lane, fix the route *description* rather than the model — then re-run and confirm the description was the bug.

**Medium — find your real concurrency limit, then use it.** Re-run `faq_parallel` with `MAX_CONCURRENT = 1`, then 4, then higher, and record the wall clock each time; the curve flattens where your provider's rate limit starts, and that number — not a guess — is your cap. Then make the fan-out survive contact with it: pass `return_exceptions=True` to the `gather`, keep the FAQs that succeeded, and report the failures instead of losing the whole batch. Finally, add a fourth worker kind, `rewrite_tone`, that restyles a reply (friendly vs formal) — and decide honestly whether it belongs in the plan as a worker or after it as a chain step. (Hint: does it depend on the other workers' output?)

**Hard — make the pipeline observable, then re-plan.** Define a `Run` Pydantic model that records the original request, the plan, every worker's input, output and duration, and the final reply; have `handle_request` return it alongside the answer and print it as JSON — one trace object per request is what makes a workflow debuggable in production (📎 Section 12's observability lesson is this idea with a UI in front of it). Then add the one thing that separates this workflow from an agent: after the workers finish, ask a model whether the plan actually covered the request, and if it did not, run a second round with the missing tasks. Cap the rounds at two — and notice that the cap is now the only thing keeping this a workflow.

## 🔑 Key Takeaways

- **One prompt with five jobs degrades on all five, invisibly.** Decomposition — not prompt polish — is the fix, and a typed schema constrains the *shape* of an output, never its *truth*.
- **Chaining** gives every step one job and makes the intermediate artifacts real Python values: printable, cacheable, testable, and swappable for a different model. Visibility is the pattern's actual product.
- **Parallelization buys latency, not cost.** Same calls, same tokens, same bill — the waiting just overlaps. Cap it with a `Semaphore` on purpose, keep dependent steps sequential inside each task, and use `return_exceptions=True` the moment it runs unattended.
- **Routing is one typed classification call plus `if`/`else`** — cheap, auditable, and fixable by rewriting a route description. 📎 Section 9's RAG router is this exact pattern applied to knowledge bases and models.
- **Orchestrator-workers writes the plan at runtime**: a Pydantic-typed task list validated before any worker runs, workers fanned out in parallel, one synthesizer producing the human-facing answer. It costs N+2 calls and has no feedback loop — use it when the subtasks genuinely differ per request.
- **Patterns compose.** Production pipelines are routers containing chains containing parallel fan-outs; picking "the" pattern is rarely the real decision.
- **The boundary that matters: a workflow is a code path you fix in advance; an agent decides its own path at runtime.** Even an LLM-written plan is still your control flow. Build the workflow first; reach for the agent only when the path cannot be known ahead of time.
- ⏭️ *Next: the agent loop itself. Then Section 12, where LangGraph turns all four of these patterns into graph edges with persistence, streaming, and human-in-the-loop built in.*